# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

In [1]:
!/opt/homebrew/bin/python3 -m venv venv
!git config --global user.email "nsknaaa@gmail.com"
!git config --global user.name "Ekaterina Neyaskina"
!git init
!source venv/bin/activate && pip install numpy scikit-learn pandas
!source venv/bin/activate && pip freeze > requirements.txt

Initialized empty Git repository in /Users/friemari/Desktop/MIPT/2 semester/Развертывание ML моделей/homework/hww7/.git/
  Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached scipy-1.17.1-cp314-cp314-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-2.4.4-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-macosx_12_0_arm64.whl (8.1 MB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached 

In [2]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
import pickle
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

hyperparameters = {"n_estimators": 100, "random_state": 42}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Точность accuracy: {accuracy:.4f}')

Writing ml_pipeline.py


### Проверяем работоспособность пайплайна

In [3]:
!source venv/bin/activate && python ml_pipeline.py

Точность accuracy: 1.0000


In [7]:
# Создаю GitHub Actions workflow (вместо .gitlab-ci.yml)
# так как в GitLab невозможно подтвердить аккаунт российским номером телефона
# а без подтверждения невозможно запутить пайплайны


In [9]:
%%writefile ml-pipeline.yml
name: ML Pipeline CI/CD

on:
  push:
    branches: [ main, master ]
  workflow_dispatch:

jobs:
  build-and-test:
    runs-on: ubuntu-latest
    
    steps:
      - name: Checkout code
        uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.10'
      
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      
      - name: Train model
        run: python ml_pipeline.py
      
      - name: Display accuracy
        run: cat accuracy.txt
      
      - name: Upload artifacts
        uses: actions/upload-artifact@v4
        with:
          name: model-artifacts
          path: |
            model.pkl
            accuracy.txt

Writing ml-pipeline.yml


In [10]:
%%writefile .gitignore
venv/
__pycache__/
*.pyc
.ipynb_checkpoints/
.DS_Store

Writing .gitignore


In [11]:
!git add .
!git commit -m "feat: ML pipeline с GitHub Actions"
!git remote add origin https://github.com/friemari/mlops_mipt_7.git
!git push -u origin main --force

[main (root-commit) 791b5fa] feat: ML pipeline с GitHub Actions
 5 files changed, 818 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 "HW7_CICD_\320\235\320\265\321\217\321\201\320\272\320\270\320\275\320\260_\320\225\320\272\320\260\321\202\320\265\321\200\320\270\320\275\320\260.ipynb"
 create mode 100644 ml-pipeline.yml
 create mode 100644 ml_pipeline.py
 create mode 100644 requirements.txt
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 8 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 7.34 KiB | 7.34 MiB/s, done.
Total 7 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/friemari/mlops_mipt_7.git
 * [new branch]      main -> main
branch 'main' set up to track 'origin/main'.


### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

```
```

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

In [ ]:
%%writefile docker-compose.yaml


## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

In [ ]:
%%writefile ml_pipeline.py
import numpy as np
import pandas as pd
import pickle
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

hyperparameters = {"n_estimators": 100, "random_state": 42}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(**hyperparameters)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Точность accuracy: {accuracy:.2f}')

# Сохраним модель и метрики
with open('model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'accuracy': accuracy, 'hyperparams': hyperparameters}, f)

with open('accuracy.txt', 'w') as f:
    f.write(f'{accuracy:.4f}')

Overwriting ml_pipeline.py


Проверяем работоспособность пайплайна

In [ ]:
!python ml_pipeline.py

Точность аccuracy: 1.00


Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

In [ ]:
%%writefile ci.yml
name: CI
on: [push, pull_request]

jobs:
  build:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v2
    - name: Set up Python
      uses: actions/setup-python@v2
      with:
        python-version: '3.x'
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install scikit-learn numpy pandas
    - name: Run pipeline
      run: |
        python -c "import numpy as np; import pandas as pd; from sklearn.datasets import load_iris; from sklearn.model_selection import train_test_split; from sklearn.ensemble import RandomForestClassifier; from sklearn.metrics import accuracy_score; iris = load_iris(); X = iris.data; y = iris.target; X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42); model = RandomForestClassifier(n_estimators=100, random_state=42); model.fit(X_train, y_train); y_pred = model.predict(X_test); accuracy = accuracy_score(y_test, y_pred); print(f'Accuracy: {accuracy:.2f}')"
    - name: Make pipeline reproducible
      run: |
        python -c "print('какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн?')"


Writing ci.yml


Копируем ci.yml в правильную директорию .github/workflows

In [ ]:
!mkdir -p .github/workflows
!mv ci.yml ./.github/workflows/ci.yml

Начинаем отправку в репозиторий

In [ ]:
!git add ./.github/workflows/ci.yml ml_pipeline.py
!git commit  -m "build(ml_pipeline.py) добавлен пайплайн GitHub Actions"
!git log

[master (root-commit) c29e765] build(ml_pipeline.py) добавлен пайплайн
 2 files changed, 35 insertions(+)
 create mode 100644 .github/workflows/ci.yml
 create mode 100644 ml_pipeline.py
commit c29e7659975b52391b4a07c33a329893f1f8426b (HEAD -> master)
Author: Your Name <you@example.com>
Date:   Wed Oct 8 19:13:08 2025 +0000

    build(ml_pipeline.py) добавлен пайплайн


После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

```
```

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.

